# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading, exploring, and processing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is described and accessible via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This provides access to the dataset's metadata and structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Let's enumerate available record sets, their unique identifiers (`@id`), and the fields (columns) within each. This gives an overview of the dataset's tabular structure for further exploration.

We'll use the Croissant model to list all record sets and (for each) their column IDs and field names.

In [ ]:
# Collect all record sets from dataset metadata (by @id)
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in Croissant metadata. The dataset may only provide inline data or documentation.")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set.id}")
        print(f"  name: {record_set.name}")
        print(f"  description: {record_set.description}")
        print("  Fields / Columns:")
        for field in record_set.fields:
            print(f"    Field @id: {field.id}")
            print(f"      name: {field.name}")
            print(f"      dataType: {field.data_type}")
        print('-'*40)

## 3. Data Extraction
Let's extract tabular data from the main record set for data analysis.

We'll reference record sets, fields, and columns **by their `@id` fields** for clarity and reproducibility.

In [ ]:
# If the dataset contains record sets, extract all tables by @id
dataframes = {}
record_set_ids = [rs.id for rs in getattr(dataset.metadata, "record_sets", [])]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# For demonstration, print columns for the first record set, if any
if record_set_ids:
    main_record_set = record_set_ids[0]
    print(f"Main RecordSet @id: {main_record_set}")
    print(f"Column IDs / Field names: {list(dataframes[main_record_set].columns)}")
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Now, let's perform some basic data processing:
- Filtering records on a numeric field
- Normalizing the numeric field
- Grouping by a categorical field (if present)

Replace `<numeric_field_id>` and `<group_field_id>` with the actual field IDs you want to analyze.

In [ ]:
# Select main dataframe
df = dataframes.get(main_record_set)

# Choose a numeric field by its @id (e.g., 'phq9_total_score' or 'gad7_total_score')
# You may have to adjust these IDs based on actual fields listed above
numeric_field = None
for col in df.columns:
    if 'phq9' in col or 'gad7' in col or 'score' in col:
        numeric_field = col
        break
if not numeric_field:
    numeric_field = df.select_dtypes('number').columns[0]  # fallback to first numeric column
print(f"Analyzing numeric field: {numeric_field}")

# Filter for high scores (e.g., scores > 10)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Records with {numeric_field} > {threshold}: {len(filtered_df)}")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First few normalized values of {numeric_field}:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a group field (e.g., demographic field)
group_field = None
for candidate in ['gender', 'level_of_education', 'village', 'marital_status', 'age_group']:
    if candidate in df.columns:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Average {numeric_field} by {group_field}:")
    display(grouped_df)
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its average grouped by a demographic attribute, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field], bins=20, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# Barplot of group mean if group field is available
if group_field:
    plt.figure(figsize=(7,4))
    order = df[group_field].dropna().unique()
    sns.barplot(data=df, x=group_field, y=numeric_field, estimator='mean', order=order)
    plt.title(f'Average {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
We have loaded and explored a survey dataset on mental health from Kilifi County, Kenya, using the `mlcroissant` library.

**Key findings:**
- The dataset provides demographic and psychometric indicators (e.g., GAD-7, PHQ-9 scores).
- We extracted records, filtered notable groups (e.g., high scores), normalized the data, and visualized distributions.
- See the Croissant metadata for additional context such as collection methods and limitations.

*You can build further on these steps for advanced analyses, e.g., predictive modeling or longitudinal assessments!*
